# 10 · Recency weighting with a surge gate — when to let recent months speak louder

Gradient-boosted trees **cannot predict above the range they were trained on**, and a model fitted on five
years of history anchors its quantiles to the old level. When a product group steps up (regulation-driven
demand, a price-led surge) the upper quantiles stay capped and every month of the surge is under-forecast.
The cheapest lever is to weight recent months more when the model is fitted: an exponential **half-life**
`w = 0.5^(months_ago / halflife)`. This notebook tests that lever properly and encodes the lessons:

| lesson | how it shows up below |
|---|---|
| recency helps series with a **sustained level step-up**, hurts seasonal and spiky ones | the A/B is run per cluster and per series, not on the portfolio total |
| a hard training **window** is worse than a soft half-life | `window18` scheme |
| the **raw size of a surge is not the predictor** of benefit — a one-off spike also looks like a surge | per-series scatter of benefit vs surge ratio |
| therefore **gate** the weighting: apply it only where the step-up is sustained | `surge_gate()` = recent-6 mean ≥ `SURGE_MIN_RATIO` × prior-year mean **and** recent-3 ≥ 0.8 × recent-6 |
| LightGBM's quantile objective misbehaves with un-normalised sample weights | every scheme normalises weights to mean 1 |

| | |
|---|---|
| **Reads** | `<scenario>_clustered` (notebook 08), `<scenario>_quantile_config` (notebook 09, the committed blends) |
| **Writes** | `<scenario>_recency_ab` — WMAPE / bias per cluster × scheme, and the recommended scheme per cluster |

## 1. Imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore")
print("✅ Libraries imported")

## 2. Configuration

In [ ]:
SCENARIO = "durable_goods_monthly"
UID, DATE_VAR, TARGET, CLUSTER_COL = "unique_id", "ds", "y", "profile_cluster"
CATS = ["product_group", "region", "channel"]
SEASON, RANDOM_STATE = 12, 42
EXTRAP_WINDOW = 6
N_VAL_QUARTERS = 3
TOP_SHARE = 0.80

# weighting schemes under test
SCHEMES = {"uniform": None, "hl6": ("halflife", 6), "hl9": ("halflife", 9), "hl12": ("halflife", 12),
           "window18": ("window", 18), "hl9_gated": ("gated", 9)}
SURGE_MIN_RATIO = 1.30        # recent-6 mean >= 1.30 x prior-year mean  -> sustained step-up candidate
SURGE_MIN_MONTHS = 18         # history needed to judge a step-up
RECENCY_ELIGIBLE = ["strategic", "smooth", "erratic"]   # clusters tested (the recommendation may exclude some)

LGBM_BASE = {"objective": "quantile", "boosting_type": "gbdt", "verbose": -1, "n_jobs": -1, "random_state": RANDOM_STATE}
PARAMS = {"strategic": {"n_estimators": 250, "learning_rate": 0.05, "num_leaves": 3, "max_depth": 4, "min_child_samples": 10,
                        "reg_alpha": 0.5, "reg_lambda": 2.5, "subsample": 0.7, "subsample_freq": 1, "colsample_bytree": 0.9, "min_split_gain": 0.05},
          "smooth": {"n_estimators": 500, "learning_rate": 0.02, "num_leaves": 15, "min_child_samples": 20,
                     "reg_alpha": 2.0, "reg_lambda": 10.0, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8}}
PARAMS["erratic"] = dict(PARAMS["smooth"])

LAKEHOUSE_NAME, LOCAL_FOLDER = "ts_forecasting", SCENARIO
root_path = Path.cwd().parent.parent
local_dir = root_path / "data" / LOCAL_FOLDER
CLUSTERED_TABLE, CONFIG_TABLE, OUT_TABLE = f"{SCENARIO}_clustered", f"{SCENARIO}_quantile_config", f"{SCENARIO}_recency_ab"


def load_table(name):
    try:
        df = spark.table(f"{LAKEHOUSE_NAME}.{name}").toPandas()
        print(f"✅ Loaded {LAKEHOUSE_NAME}.{name} ({len(df):,} rows)")
    except Exception as e:
        df = pd.read_parquet(local_dir / f"{name}.parquet")
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); loaded {local_dir / (name + '.parquet')} ({len(df):,} rows)")
    return df


def save_table(df, name):
    try:
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{LAKEHOUSE_NAME}.{name}")
        print(f"✅ Saved {len(df):,} rows to {LAKEHOUSE_NAME}.{name}")
    except Exception as e:
        local_dir.mkdir(parents=True, exist_ok=True)
        df.to_parquet(local_dir / f"{name}.parquet", index=False)
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); saved {len(df):,} rows to {local_dir / (name + '.parquet')}")


print(f"✅ Config | schemes {list(SCHEMES)} | gate: recent-6 >= {SURGE_MIN_RATIO}x prior year & recent-3 >= 0.8 x recent-6")

## 3. Load the panel and the committed recipe; rebuild the notebook-09 engine (compact)

The blends are **fixed to the committed recipe** from notebook 09, so the A/B isolates the effect of the
weights. Only the quantiles the blends need are trained.

In [ ]:
df = load_table(CLUSTERED_TABLE)
df[DATE_VAR] = pd.to_datetime(df[DATE_VAR])
df = df.sort_values([UID, DATE_VAR]).drop_duplicates([UID, DATE_VAR], keep="last").reset_index(drop=True)
cfg = json.loads(load_table(CONFIG_TABLE)["json"].iloc[0])
COMMITTED = cfg["committed"]
FY_START_MONTH = int(cfg.get("fy_start_month", 10))
BLEND_Q = sorted({q for c in COMMITTED.values() for q in (c["blend"]["q_lo"], c["blend"]["q_hi"])})
TUNE_CLUSTERS = [c for c in COMMITTED if c in RECENCY_ELIGIBLE]
DATA_MAX = df[DATE_VAR].max()
print(f"committed blends: { {c: v['blend'] for c, v in COMMITTED.items()} } -> quantiles trained {BLEND_Q}")


def fiscal_parts(ts):
    fm = ((ts.month - FY_START_MONTH) % 12) + 1
    return ts.year + (1 if ts.month >= FY_START_MONTH else 0), (fm - 1) // 3 + 1, fm


def fq_label(ts):
    fy, fq, _ = fiscal_parts(ts)
    return f"FY{fy % 100:02d}-Q{fq}"


def calendar_features(dates):
    fm = ((dates.dt.month - FY_START_MONTH) % 12) + 1
    return pd.DataFrame({"fiscal_month": fm, "fiscal_quarter": (fm - 1) // 3 + 1, "quarter_end": fm.isin([3, 6, 9, 12]).astype(int),
                         "fy_end": (fm == 12).astype(int), "days_in_month": dates.dt.days_in_month}, index=dates.index)


CAL_FEATURES = ["fiscal_month", "fiscal_quarter", "quarter_end", "fy_end", "days_in_month"]
MODEL_FEATURES = CAL_FEATURES + ["y_lag2", "y_lag3", "y_lag12"] + CATS
CAT_LEVELS = {c: sorted(df[c].dropna().astype(str).unique()) for c in CATS}

feat = df.copy()
feat[CAL_FEATURES] = calendar_features(feat[DATE_VAR])
g = feat.groupby(UID, sort=False)
for k in (2, 3, 12):
    feat[f"y_lag{k}"] = g[TARGET].shift(k)
if "price_per_unit" not in feat.columns:
    feat["price_per_unit"] = feat[TARGET] / feat["quantity"].replace(0, np.nan)
for c in CATS:
    feat[c] = pd.Categorical(feat[c].astype(str), categories=CAT_LEVELS[c])


def month_in_quarter(ts):
    return ((fiscal_parts(ts)[2] - 1) % 3) + 1


def quarter_start(ts):
    return ts.replace(day=1) - pd.DateOffset(months=month_in_quarter(ts) - 1)


def make_val_folds(data_max, n_val):
    cur, miq = quarter_start(data_max), month_in_quarter(data_max)
    last_complete = cur if miq == 3 else cur - pd.DateOffset(months=3)
    folds = []
    for i in range(n_val, 0, -1):
        qs = last_complete - pd.DateOffset(months=3 * (i - 1))
        folds.append({"name": f"fold{n_val - i + 1}", "quarter": fq_label(qs), "origin": qs - pd.DateOffset(months=1),
                      "months": [qs + pd.DateOffset(months=j) for j in range(3)]})
    return folds


VAL_FOLDS = make_val_folds(DATA_MAX, N_VAL_QUARTERS)
for f in VAL_FOLDS:
    print(f"  {f['name']} | {f['quarter']} | origin {f['origin'].date()}")

_rev = feat.groupby(UID, observed=True)[TARGET].sum().sort_values(ascending=False)
FOCUS = set(_rev.index[(_rev.cumsum() / _rev.sum()).shift(fill_value=0.0) < TOP_SHARE])

In [ ]:
def qcol(q):
    return f"q{int(round(q * 100))}"


def _extrapolate(series, upto):
    s = series.dropna().sort_index()
    if s.empty:
        return pd.Series(dtype=float)
    diffs = np.diff(s.iloc[-EXTRAP_WINDOW:].to_numpy(float))
    step = float(np.median(diffs)) if len(diffs) else 0.0
    idx = pd.date_range(s.index.min(), max(s.index.max(), upto), freq="MS")
    path, cur = s.reindex(idx), float(s.iloc[-1])
    for m in idx[idx > s.index.max()]:
        cur = max(cur + step, 0.0)
        path.loc[m] = cur
    return path


def served_rows(cdf, months, origin):
    hist = cdf[(cdf[DATE_VAR] <= origin) & cdf[TARGET].notna()]
    out = []
    for uid, gg in hist.groupby(UID, sort=False, observed=True):
        gg = gg.sort_values(DATE_VAR)
        yv = gg.set_index(DATE_VAR)[TARGET]
        last_y, last_row = float(yv.iloc[-1]), gg.iloc[-1]
        for m in months:
            lag = lambda k: float(yv.get(m - pd.DateOffset(months=k), last_y)) if (m - pd.DateOffset(months=k)) <= origin else last_y
            out.append({UID: uid, DATE_VAR: m, CLUSTER_COL: last_row[CLUSTER_COL], **{c: last_row[c] for c in CATS},
                        "y_lag2": lag(2), "y_lag3": lag(3), "y_lag12": lag(12),
                        "snaive": float(yv.get(m - pd.DateOffset(months=SEASON), last_y))})
    rows = pd.DataFrame(out)
    rows[CAL_FEATURES] = calendar_features(rows[DATE_VAR])
    for c in CATS:
        rows[c] = pd.Categorical(rows[c].astype(str), categories=CAT_LEVELS[c])
    return rows


ACTUAL = feat.set_index([UID, DATE_VAR])[TARGET]
STRICT_ROWS = {f["name"]: {clu: served_rows(feat[feat[CLUSTER_COL] == clu], f["months"], f["origin"]) for clu in TUNE_CLUSTERS}
               for f in VAL_FOLDS}


def train_and_score(fold, clu, weight_fn):
    tr = feat[(feat[CLUSTER_COL] == clu) & (feat[DATE_VAR] <= fold["origin"])].sort_values([UID, DATE_VAR])
    w = None
    if weight_fn is not None:
        w = np.asarray(weight_fn(tr, fold["origin"]), float)
        w = w / w.mean()                                   # mean-1 scale: raw weights blow up the quantile objective
    rows = STRICT_ROWS[fold["name"]][clu].copy()
    for q in BLEND_Q:
        m = lgb.LGBMRegressor(**{**LGBM_BASE, "alpha": q, **PARAMS[clu]}).fit(tr[MODEL_FEATURES], tr[TARGET], sample_weight=w, categorical_feature=CATS)
        rows[qcol(q)] = m.predict(rows[MODEL_FEATURES])
    cols = [qcol(q) for q in BLEND_Q]
    rows[cols] = np.sort(rows[cols].to_numpy(float), axis=1)
    b = COMMITTED[clu]["blend"]
    rows["yhat"] = b["w"] * rows[qcol(b["q_lo"])] + (1 - b["w"]) * rows[qcol(b["q_hi"])]
    rows["y"] = [ACTUAL.get((u, d), np.nan) for u, d in zip(rows[UID], rows[DATE_VAR])]
    rows["fold"], rows["quarter"] = fold["name"], fold["quarter"]
    return rows[rows["y"].notna()]


def wmape(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    d = np.abs(a).sum()
    return np.nan if d == 0 else 100.0 * np.abs(a - p).sum() / d


def bias_pct(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    d = np.abs(a).sum()
    return np.nan if d == 0 else 100.0 * (p - a).sum() / d


print("✅ compact engine ready (strict rows prebuilt per fold x cluster)")

## 4. Weighting schemes and the surge gate

All schemes return one weight per training row (the frame is sorted by series, then month):

- **halflife(h)**: `0.5^(months_ago / h)` for every series.
- **window(n)**: 1 for the last `n` months, ~0 before — the hard cut the field test showed to be the worst option.
- **gated(h)**: the half-life weights **only for series whose level stepped up and stayed up**; every other
  series keeps uniform weights. Step-up = mean of the last 6 months ≥ `SURGE_MIN_RATIO` × mean of the 12 months
  before that, **and** mean of the last 3 months ≥ 0.8 × mean of the last 6 (a spike that already reverted does
  not qualify).

In [ ]:
def months_ago(tr, origin):
    return ((origin.year - tr[DATE_VAR].dt.year) * 12 + (origin.month - tr[DATE_VAR].dt.month)).clip(lower=0).to_numpy(float)


def halflife_weights(tr, origin, h):
    return np.power(0.5, months_ago(tr, origin) / h)


def window_weights(tr, origin, n):
    return np.where(months_ago(tr, origin) < n, 1.0, 1e-3)


def surge_flags(tr, origin, ratio=SURGE_MIN_RATIO, min_months=SURGE_MIN_MONTHS):
    """{series: True/False} - sustained level step-up as of the origin (leak-free: tr holds rows <= origin only)."""
    flags = {}
    for uid, gg in tr.groupby(UID, sort=False, observed=True):
        yv = gg.sort_values(DATE_VAR)[TARGET].to_numpy(float)
        if len(yv) < min_months:
            flags[uid] = False
            continue
        recent6, recent3, prior = np.nanmean(yv[-6:]), np.nanmean(yv[-3:]), np.nanmean(yv[-18:-6])
        flags[uid] = bool(prior > 0 and recent6 / prior >= ratio and recent3 >= 0.8 * recent6)
    return flags


def gated_halflife_weights(tr, origin, h):
    flags = surge_flags(tr, origin)
    w = halflife_weights(tr, origin, h)
    gated = tr[UID].map(flags).fillna(False).to_numpy(bool)
    return np.where(gated, w, 1.0)


def make_weight_fn(spec):
    if spec is None:
        return None
    kind, p = spec
    return {"halflife": lambda tr, o: halflife_weights(tr, o, p), "window": lambda tr, o: window_weights(tr, o, p),
            "gated": lambda tr, o: gated_halflife_weights(tr, o, p)}[kind]


# how many series does the gate flag at each origin?
for f in VAL_FOLDS + [{"name": "served", "origin": DATA_MAX}]:
    tr = feat[feat[DATE_VAR] <= f["origin"]]
    fl = {clu: sum(surge_flags(tr[tr[CLUSTER_COL] == clu], f["origin"]).values()) for clu in TUNE_CLUSTERS}
    print(f"  gate @ {f['origin'].date()} ({f['name']}): series flagged as sustained step-up -> {fl}")

## 5. A/B per cluster × scheme (walk-forward, as served, revenue-focus series)

In [ ]:
results, detail = [], []
for clu in TUNE_CLUSTERS:
    for name, spec in SCHEMES.items():
        frames = [train_and_score(f, clu, make_weight_fn(spec)) for f in VAL_FOLDS]
        s = pd.concat(frames, ignore_index=True)
        s["scheme"] = name
        detail.append(s)
        focus = s[s[UID].astype(str).isin(FOCUS)] if clu != "strategic" else s
        per_fold = {f["quarter"]: wmape(focus.loc[focus["fold"] == f["name"], "y"], focus.loc[focus["fold"] == f["name"], "yhat"]) for f in VAL_FOLDS}
        results.append({"cluster": clu, "scheme": name, "wmape": wmape(focus["y"], focus["yhat"]), "bias_pct": bias_pct(focus["y"], focus["yhat"]),
                        **{f"wmape_{k}": v for k, v in per_fold.items()}})
    print(f"  {clu}: done")
ab = pd.DataFrame(results)
ab["delta_vs_uniform"] = ab["wmape"] - ab.groupby("cluster")["wmape"].transform(lambda s: s[ab.loc[s.index, "scheme"] == "uniform"].iloc[0])
detail = pd.concat(detail, ignore_index=True)
print("\nWMAPE (%) per cluster x weighting scheme - strict rows, pooled over the validation folds:")
print(ab.pivot(index="scheme", columns="cluster", values="wmape").loc[list(SCHEMES)].round(1).to_string())
print("\nnet bias (%):")
print(ab.pivot(index="scheme", columns="cluster", values="bias_pct").loc[list(SCHEMES)].round(1).to_string())

## 6. Per-series view — who benefits, and why the raw surge size is a poor predictor

For every series: the WMAPE change `hl9 − uniform` (negative = recency helped) against the series' surge
ratio at the last origin (recent-6 mean ÷ prior-year mean). Sustained step-ups sit to the right **and**
below zero; one-off spikes sit to the right but above zero — which is what the gate is for.

In [ ]:
piv = (detail.groupby(["scheme", UID], observed=True).apply(lambda s: wmape(s["y"], s["yhat"]), include_groups=False)
       .unstack("scheme"))
piv = piv.dropna(subset=["uniform", "hl9"])
piv["delta_hl9"] = piv["hl9"] - piv["uniform"]
last_tr = feat[feat[DATE_VAR] <= VAL_FOLDS[-1]["origin"]]
ratios, flags = {}, {}
for uid, gg in last_tr.groupby(UID, sort=False, observed=True):
    yv = gg.sort_values(DATE_VAR)[TARGET].to_numpy(float)
    prior = np.nanmean(yv[-18:-6]) if len(yv) >= 18 else np.nan
    ratios[uid] = np.nanmean(yv[-6:]) / prior if prior and prior > 0 else np.nan
flags = surge_flags(last_tr[last_tr[CLUSTER_COL].isin(TUNE_CLUSTERS)], VAL_FOLDS[-1]["origin"])
piv["surge_ratio"] = piv.index.map(ratios)
piv["gated"] = piv.index.map(flags).fillna(False).astype(bool)
piv["cluster"] = piv.index.map(feat.groupby(UID, observed=True)[CLUSTER_COL].first())
piv = piv[piv.index.isin(FOCUS) | (piv["cluster"] == "strategic")]
helped = piv.groupby("cluster")["delta_hl9"].apply(lambda s: f"{(s < 0).mean():.0%} of series helped (median delta {s.median():+.1f} pts)")
print("hl9 vs uniform, revenue-focus series:\n" + helped.to_string())
print(f"\ncorrelation(surge ratio, benefit of recency): {piv[['surge_ratio', 'delta_hl9']].dropna().corr().iloc[0, 1]:+.2f}  "
      f"(a weak or positive number = the raw surge size does not predict who benefits)")
print(f"gated series: {int(piv['gated'].sum())} | median delta where gated {piv.loc[piv['gated'], 'delta_hl9'].median():+.1f} pts "
      f"vs not gated {piv.loc[~piv['gated'], 'delta_hl9'].median():+.1f} pts")

fig, ax = plt.subplots(figsize=(9, 5))
for clu, g in piv.groupby("cluster"):
    ax.scatter(g["surge_ratio"], g["delta_hl9"], s=np.where(g["gated"], 80, 25), alpha=0.7, label=f"{clu} (gated = large marker)")
ax.axhline(0, color="k", linewidth=1)
ax.axvline(SURGE_MIN_RATIO, color="red", linestyle="--", linewidth=1, label=f"gate ratio {SURGE_MIN_RATIO}")
ax.set(xlabel="surge ratio at the last origin (recent-6 mean / prior-year mean)", ylabel="WMAPE change, hl9 - uniform (pts)",
       title="Recency benefit per series vs how much the series surged", xscale="log")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 7. Recommendation per cluster and the record

The recommended scheme per cluster is the one with the lowest strict WMAPE **that does not worsen the bias by
more than a few points** — and `uniform` wins ties. Clusters where every weighting scheme loses to `uniform`
are excluded from recency altogether (the field result for spiky clusters).

In [ ]:
rec = []
for clu, g in ab.groupby("cluster"):
    base = g[g["scheme"] == "uniform"].iloc[0]
    cand = g[(g["wmape"] < base["wmape"] - 0.5) & (g["bias_pct"].abs() <= abs(base["bias_pct"]) + 5)]
    pick = cand.sort_values("wmape").iloc[0] if len(cand) else base
    rec.append({"cluster": clu, "recommended_scheme": pick["scheme"], "uniform_wmape": base["wmape"], "recommended_wmape": pick["wmape"],
                "gain_pts": base["wmape"] - pick["wmape"], "recommended_bias": pick["bias_pct"]})
rec = pd.DataFrame(rec)
print(rec.round(1).to_string(index=False))
save_table(ab.assign(data_through=str(DATA_MAX.date())).merge(rec[["cluster", "recommended_scheme"]], on="cluster"), OUT_TABLE)

## ✅ CHECK POINT with the data scientist

- Adopt recency **per cluster**, never globally: wire `weight_fn = gated_halflife_weights(h)` into
  `train_cluster_quantiles` of notebook 09 for the recommended clusters only (the hook already exists).
- The gate is evaluated at every origin from history ≤ origin, so it is leak-free and re-evaluates itself
  every run — a series drops out of recency once its step-up reverts.
- If a cluster still under-forecasts a steep surge after recency, the cause is the tree ceiling, not the weights:
  look at an aggregate anchor ([11](11-Aggregate-Anchor-Reconciliation)) or an extrapolating model family.